# Data Preparation

In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from config import (
  UNDERSTANDING_REGENCIES_CSV,
  FEATURE_EVALUATION_JSON,
  FEATURE_SELECTION_JSON,
  PREPARED_REGENCIES_CSV
)

In [2]:
MAX_VIF = 10.0
MIN_CV = 10.0
SKEWNESS_THRESHOLD = 2.0

In [3]:
print(f"Parameter MAX_VIF            : {MAX_VIF}")
print(f"Parameter MIN_CV             : {MIN_CV}%")
print(f"Parameter SKEWNESS_THRESHOLD : {SKEWNESS_THRESHOLD}")

Parameter MAX_VIF            : 10.0
Parameter MIN_CV             : 10.0%
Parameter SKEWNESS_THRESHOLD : 2.0


In [4]:
df_reg = pd.read_csv(UNDERSTANDING_REGENCIES_CSV)
print(df_reg.head().to_markdown(index=False))

|   province_id |   regency_no | regency_name   |   total_koperasi |   koperasi_nib |   koperasi_npwp |   koperasi_rat |   simpanan_pokok |   simpanan_wajib |   volume_transaksi |   nilai_transaksi |
|--------------:|-------------:|:---------------|-----------------:|---------------:|----------------:|---------------:|-----------------:|-----------------:|-------------------:|------------------:|
|            17 |            1 | KAB. BADUNG    |               64 |             39 |              63 |             59 |       2.735e+07  |        3.43e+06  |              65091 |       2.03328e+08 |
|            17 |            2 | KAB. BANGLI    |               72 |             37 |              72 |             72 |       2.0123e+08 |        5.141e+07 |                  0 |       0           |
|            17 |            3 | KAB. BULELENG  |              148 |             97 |             147 |            121 |       8.4915e+07 |        4.425e+07 |              35197 |       4.17694e+08 |


## Imputasi Nilai Hilang (kNN Imputer)

In [5]:
num_cols = df_reg.select_dtypes('number').columns

imputer = KNNImputer(n_neighbors=5)

df_reg[num_cols] = imputer.fit_transform(df_reg[num_cols])

## Pemilihan Fitur

In [7]:
with open(FEATURE_EVALUATION_JSON, 'r', encoding='utf-8') as f:
  feature_config = json.load(f)

vif_dict = feature_config.get('vif', {})
cv_dict = feature_config.get('variability_cv', {})
skew_dict = feature_config.get('skewness', {})

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\kopdes\\data\\1_understanding\\feature_evaluation.json'

In [ ]:
selected_features = []
eliminated_features = []

for feat, vif_val in vif_dict.items():
  cv_val = cv_dict.get(feat, 100.0)
  
  if vif_val <= MAX_VIF and cv_val >= MIN_CV:
    selected_features.append(feat)
  else:
    eliminated_features.append(feat)

log_transform_features = []

for feat in selected_features:
  skew_val = skew_dict.get(feat, 0)
  
  if skew_val >= SKEWNESS_THRESHOLD:
    log_transform_features.append(feat)

print(f"Fitur Terpilih         : {selected_features}")
print(f"Fitur Dieliminasi      : {eliminated_features}")
print(f"Fitur Transformasi Log : {log_transform_features}")

## Transformasi & Standardisasi Fitur

### Transformasi Logaritmik (Log1p)

In [ ]:
features_present = []

for col in selected_features:
  if col in df_reg.columns:
    features_present.append(col)

X_df = df_reg[features_present].copy()

for col in log_transform_features:
  if col in X_df.columns:
    X_df[col] = np.log1p(X_df[col].clip(lower=0))

print(X_df.describe().T.to_markdown())

### Standardisasi Fitur (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values)

scaled_cols = []

for c in features_present:
  scaled_cols.append(f"scaled_{c}")

df_reg[scaled_cols] = X_scaled

print(f"Total Kolom Fitur Terstandarisasi Ditambahkan: {len(scaled_cols)}")

## Penyimpanan Hasil

In [ ]:
df_reg.to_csv(PREPARED_REGENCIES_CSV, index=False)
print(f"File Prepared Kab/Kota disimpan di       : {PREPARED_REGENCIES_CSV}")

In [ ]:
feature_selection = {
  "selected_features": selected_features,
  "eliminated_features": eliminated_features,
  "log_transform_features": log_transform_features,
  "scaled_feature_columns": scaled_cols
}

with open(FEATURE_SELECTION_JSON, 'w', encoding='utf-8') as f:
  json.dump(feature_selection, f, indent=2)

print(f"Hasil Feature Selection disimpan di      : {FEATURE_SELECTION_JSON}")